# By using lorentzian function we can get a clean spectra looking function in which we add randomness and noise in order to create entry for our dataset

In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNKCIE
# ─────────────────────────────────────────────────────────────────────────────

def slugify(name: str) -> str:
    """Premení názov zlúčeniny na bezpečný názov priečinka."""
    name = name.strip().lower()
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^a-z0-9_\-\.]", "", name)
    return name or "unknown"


def lorentzian(x: np.ndarray, center: float, hwhm: float) -> np.ndarray:
    """
    Lorentzova funkcia — tvar jedného NMR píku.
    
    Parametre:
        x      : os ppm hodnôt
        center : poloha píku v ppm
        hwhm   : pološírka píku (Half Width at Half Maximum) v ppm
    """
    return (hwhm ** 2) / ((x - center) ** 2 + hwhm ** 2)


def resample_to(vector: np.ndarray, target_len: int = 1024) -> np.ndarray:
    """
    Resampľuje vektor na požadovanú dĺžku pomocou lineárnej interpolácie.
    Používame to aby PNG aj .npy mali rovnaký počet bodov.
    """
    x_old = np.linspace(0, 1, len(vector))
    x_new = np.linspace(0, 1, target_len)
    return np.interp(x_new, x_old, vector).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# J-COUPLING — rozštiepenie píkov na multiplet
# ─────────────────────────────────────────────────────────────────────────────

def get_multiplet_lines(center_ppm: float, n_neighbors: int, j_hz: float, spectrometer_mhz: float = 400.0):
    """
    Vypočíta polohy a relatívne intenzity čiar multipletu.

    Čo je J-coupling?
        V reálnom NMR nie je pík jeden ostrý bod — susedné vodíkové atómy
        na molekule navzájom ovplyvňujú svoje signály. Výsledkom je že
        jeden pík sa rozštiepí na viac čiar (multiplet):
            - 0 susedov → singlet  (1 čiara,        intenzity: [1])
            - 1 sused   → dublet   (2 čiary,         intenzity: [1, 1])
            - 2 susedia → triplet  (3 čiary,         intenzity: [1, 2, 1])
            - 3 susedia → kvartet  (4 čiary,         intenzity: [1, 3, 3, 1])
            - atď. podľa Pascalovho trojuholníka

    Parametre:
        center_ppm       : stredná poloha píku v ppm
        n_neighbors      : počet susedných H atómov (určuje typ multipletu)
        j_hz             : veľkosť J-couplingu v Hz (typicky 1–15 Hz pre 1H)
        spectrometer_mhz : frekvencia spektrometra v MHz (štandard = 400 MHz)
    
    Vracia:
        positions   : polohy jednotlivých čiar v ppm
        intensities : relatívne intenzity čiar (z Pascalovho trojuholníka)
    """

    # Pascalov trojuholník — relatívne intenzity čiar multipletu
    # Riadok 0 = singlet [1], riadok 1 = dublet [1,1], atď.
    pascal = [
        [1],
        [1, 1],
        [1, 2, 1],
        [1, 3, 3, 1],
        [1, 4, 6, 4, 1],
        [1, 5, 10, 10, 5, 1],
    ]

    # Ak máme viac susedov ako máme v tabuľke, použijeme singlet
    n = min(n_neighbors, len(pascal) - 1)
    relative_intensities = np.array(pascal[n], dtype=np.float32)

    # Normalizujeme intenzity tak aby súčet = 1 (zachováme celkovú plochu)
    relative_intensities /= relative_intensities.sum()

    # Počet čiar v multiplete
    n_lines = len(relative_intensities)

    # Rozostup medzi čiarami: J [Hz] → J [ppm] = J_hz / spectrometer_mhz
    spacing_ppm = j_hz / spectrometer_mhz

    # Polohy čiar: symetricky okolo centra
    # Napr. triplet: center - spacing, center, center + spacing
    offsets = np.linspace(
        -(n_lines - 1) / 2 * spacing_ppm,
         (n_lines - 1) / 2 * spacing_ppm,
        n_lines
    )
    positions = center_ppm + offsets

    return positions, relative_intensities


# ─────────────────────────────────────────────────────────────────────────────
# GENERÁTOR SPEKTRA
# ─────────────────────────────────────────────────────────────────────────────

def generate_spectrum(
    peaks_ppm:  np.ndarray,
    peaks_int:  np.ndarray,
    ppm_min:    float = -0.5,
    ppm_max:    float = 10.5,
    n_points:   int   = 8192,
    rng:        np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Vygeneruje jedno syntetické 1H NMR spektrum.

    Postup:
        1. Vytvorí os ppm (x)
        2. Pre každý pík zo vstupného CSV:
            a. Pridá malý náhodný shift (simuluje chybu referencovania)
            b. Rozštiepí pík na multiplet pomocou J-couplingu
            c. Každú čiaru multipletu nakreslí ako Lorentzián
        3. Pridá kontaminanty (voda, CDCl3)
        4. Pridá hladký baseline drift
        5. Pridá gaussovský šum
        6. Normalizuje výsledok na rozsah [-1, 1]

    Parametre:
        peaks_ppm : polohy píkov v ppm (zo vstupného CSV)
        peaks_int : intenzity píkov (zo vstupného CSV)
        ppm_min   : ľavý okraj spektra
        ppm_max   : pravý okraj spektra
        n_points  : počet bodov na osi x (rozlíšenie)
        rng       : numpy random generator (pre reprodukovateľnosť)
    """

    rng = np.random.default_rng() if rng is None else rng

    # ── 1. Os ppm ────────────────────────────────────────────────────────────
    x = np.linspace(ppm_min, ppm_max, n_points).astype(np.float32)
    y = np.zeros_like(x)

    # ── 2. Globálne náhodné variácie ─────────────────────────────────────────
    # Simulujeme že každé meranie je trochu iné

    # Referencing error: celé spektrum sa posunie o malý kúsok
    global_shift = rng.uniform(-0.03, 0.03)

    # Intensity scale: celkové spektrum je trochu silnejšie alebo slabšie
    global_scale = rng.uniform(0.6, 1.4)

    # ── 3. Píky s J-couplingom ───────────────────────────────────────────────
    for peak_ppm, peak_int in zip(peaks_ppm, peaks_int):

        # Malý per-peak shift + náhodný jitter intenzity
        shifted_ppm = peak_ppm + global_shift + rng.normal(0, 0.002)
        scaled_int  = peak_int * global_scale * rng.uniform(0.85, 1.15)

        # Šírka píku (HWHM) — každý pík je trochu inak ostrý
        hwhm = rng.uniform(0.0015, 0.006)

        # J-coupling: náhodne rozhodneme koľko susedov má tento vodík
        # V reálnych molekulách to závisí od štruktúry, tu to simulujeme náhodne
        # Váhy: singlet je najčastejší, vyššie multiplety menej časté
        n_neighbors = rng.choice([0, 1, 2, 3, 4], p=[0.25, 0.30, 0.25, 0.15, 0.05])

        # J-konštanta v Hz: typické hodnoty pre vicinal coupling (3J) = 6-8 Hz
        j_hz = rng.uniform(3.0, 10.0)

        # Vypočítame polohy a intenzity čiar multipletu
        line_positions, line_intensities = get_multiplet_lines(
            center_ppm       = shifted_ppm,
            n_neighbors      = n_neighbors,
            j_hz             = j_hz,
            spectrometer_mhz = 400.0,
        )

        # Každú čiaru multipletu pridáme do spektra
        for line_ppm, line_rel_int in zip(line_positions, line_intensities):
            y += scaled_int * line_rel_int * lorentzian(x, line_ppm, hwhm)

    # ── 4. Kontaminanty ──────────────────────────────────────────────────────
    # Voda (~4.70 ppm) — objavuje sa v ~70% meraní
    if rng.random() < 0.7:
        water_ppm  = 4.70 + rng.normal(0, 0.02)
        water_int  = rng.uniform(0.05, 0.2)
        water_hwhm = rng.uniform(0.002, 0.02)
        y += water_int * lorentzian(x, water_ppm, water_hwhm)

    # CDCl3 (~7.26 ppm) — bežné rozpúšťadlo
    if rng.random() < 0.7:
        cdcl3_ppm  = 7.26 + rng.normal(0, 0.01)
        cdcl3_int  = rng.uniform(0.02, 0.15)
        cdcl3_hwhm = rng.uniform(0.001, 0.01)
        y += cdcl3_int * lorentzian(x, cdcl3_ppm, cdcl3_hwhm)

    # ── 5. Baseline drift ────────────────────────────────────────────────────
    # Reálne spektrá majú pomalé vlnenie pozadia (neidealné meranie)
    baseline_amplitude = rng.uniform(0.0, 0.01)
    raw_noise          = rng.normal(0, 1, n_points)
    smoothing_kernel   = np.ones(200) / 200
    smooth_baseline    = np.convolve(raw_noise, smoothing_kernel, mode="same")
    y += baseline_amplitude * smooth_baseline

    # ── 6. Gaussovský šum ────────────────────────────────────────────────────
    noise_sigma = rng.uniform(0.0005, 0.003)
    y += rng.normal(0, noise_sigma, size=n_points).astype(np.float32)

    # ── 7. Normalizácia na [-1, 1] ───────────────────────────────────────────
    # Odčítame medián (odstránime DC offset)
    y = y - np.median(y)

    # Odrežeme extrémne outliere (napr. spike šum)
    p995 = np.percentile(np.abs(y), 99.5)
    y    = np.clip(y, -p995, p995)

    # Normalizujeme podľa maxima → rozsah [-1, 1]
    y = y / (np.max(np.abs(y)) + 1e-6)

    return x, y.astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# ULOŽENIE PNG
# ─────────────────────────────────────────────────────────────────────────────

def save_png(x: np.ndarray, y: np.ndarray, path_png: str, dpi: int = 100, linewidth: float = 4.0):
    """
    Uloží spektrum ako PNG obrázok.
    
    Dôležité detaily:
        - figsize=(10.24, 2.46) pri dpi=100 → presne 1024×246 pixelov
        - invert_xaxis: NMR konvencia — ppm rastie sprava doľava
        - subplots_adjust: graf vyplní celú plochu bez okrajov
        - Pevné y limity: zabezpečia že CV2 extrakcia bude konzistentná
    """
    fig, ax = plt.subplots(figsize=(10.24, 2.46))

    ax.plot(x, y, linewidth=linewidth, color="black")
    ax.invert_xaxis()                        # NMR konvencia: ppm rastie doprava
    ax.set_ylim(-0.2, 1.1)                   # pevné limity pre konzistenciu
    ax.set_xlim(x.min(), x.max())
    ax.axis("off")                           # bez osí — čistý obrázok

    fig.patch.set_facecolor("white")
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

    plt.savefig(path_png, dpi=dpi, facecolor="white")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# HLAVNÝ BUILDER
# ─────────────────────────────────────────────────────────────────────────────

def build_dataset(
    csv_path:            str,
    out_dir:             str  = "dataset_out",
    samples_per_compound: int  = 500,
    save_vectors:        bool = True,
    save_images:         bool = True,
    vector_len:          int  = 1024,
    seed:                int  = 0,
):
    """
    Vygeneruje celý dataset zo vstupného CSV súboru.

    Štruktúra výstupu:
        out_dir/
        ├── label_map.json       ← mapovanie názov → číslo triedy
        ├── index.csv            ← zoznam všetkých vzoriek
        ├── images/
        │   └── acetone/
        │       ├── acetone_00000.png
        │       └── ...
        └── vectors/
            └── acetone/
                ├── acetone_00000.npy
                └── ...

    CSV musí obsahovať stĺpce: compound, ppm, intensity
    """

    rng = np.random.default_rng(seed)
    df  = pd.read_csv(csv_path)

    # Validácia vstupného CSV
    required_columns = {"compound", "ppm", "intensity"}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"CSV musí obsahovať stĺpce: {required_columns} (nájdené: {set(df.columns)})")

    # Čistenie — odstránime riadky s chýbajúcimi hodnotami
    df = df.dropna(subset=["compound", "ppm", "intensity"]).copy()
    df["compound"] = df["compound"].astype(str)

    os.makedirs(out_dir, exist_ok=True)

    # Label mapa: každej zlúčenine priradíme číslo triedy
    compounds = sorted(df["compound"].unique().tolist())
    label_map = {compound: idx for idx, compound in enumerate(compounds)}

    with open(os.path.join(out_dir, "label_map.json"), "w", encoding="utf-8") as f:
        json.dump(label_map, f, ensure_ascii=False, indent=2)

    # Vytvorenie výstupných priečinkov
    if save_images:
        os.makedirs(os.path.join(out_dir, "images"),  exist_ok=True)
    if save_vectors:
        os.makedirs(os.path.join(out_dir, "vectors"), exist_ok=True)

    rows = []

    for compound, group in df.groupby("compound", sort=False):

        peaks_ppm = group["ppm"].to_numpy(np.float32)
        peaks_int = group["intensity"].to_numpy(np.float32)
        folder_name = slugify(compound)

        # Priečinky pre túto zlúčeninu
        img_dir = os.path.join(out_dir, "images",  folder_name)
        vec_dir = os.path.join(out_dir, "vectors", folder_name)

        if save_images:  os.makedirs(img_dir, exist_ok=True)
        if save_vectors: os.makedirs(vec_dir, exist_ok=True)

        for i in range(samples_per_compound):

            # Generuj spektrum
            x, y = generate_spectrum(peaks_ppm, peaks_int, rng=rng)

            # Resample na target_len — PNG aj .npy musia byť z rovnakých dát!
            y_final = resample_to(y, target_len=vector_len)
            x_final = np.linspace(x.min(), x.max(), vector_len)

            sample_id = f"{folder_name}_{i:05d}"
            png_path  = ""
            npy_path  = ""

            if save_images:
                png_path = os.path.join(img_dir, f"{sample_id}.png")
                save_png(x_final, y_final, png_path)

            if save_vectors:
                npy_path = os.path.join(vec_dir, f"{sample_id}.npy")
                np.save(npy_path, y_final)

            rows.append({
                "sample_id": sample_id,
                "compound":  compound,
                "label":     label_map[compound],
                "png_path":  png_path,
                "npy_path":  npy_path,
            })

        print(f"Hotovo: {compound} → {samples_per_compound} vzoriek")

    # Uloženie indexu
    index_df = pd.DataFrame(rows)
    index_df.to_csv(os.path.join(out_dir, "index.csv"), index=False)
    print(f"\nDataset uložený do: {out_dir}")
    print(f"Celkovo vzoriek: {len(rows)}")
    print(f"Zlúčeniny ({len(compounds)}): {compounds}")


# ─────────────────────────────────────────────────────────────────────────────
# SPUSTENIE
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH = "../spectra_dataset.csv"
OUT_DIR  = "../nmr_dataset"

build_dataset(
    csv_path             = CSV_PATH,
    out_dir              = OUT_DIR,
    samples_per_compound = 50,
    save_vectors         = True,
    save_images          = True,
    vector_len           = 1024,
    seed                 = 42,
)

Hotovo: ethanol → 50 vzoriek
Hotovo: acetone → 50 vzoriek
Hotovo: toluene → 50 vzoriek

Dataset uložený do: ../nmr_dataset
Celkovo vzoriek: 150
Zlúčeniny (3): ['acetone', 'ethanol', 'toluene']
